# m51 lab · Cardiac CT for EP: from line integrals to wall thickness

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

You compute Hounsfield units from tabulated attenuation, project a synthetic thorax slice and check the
central-slice theorem, rebuild the slice by filtered back-projection, add quantum noise and watch it fall with
dose, work out what the gating window does to a moving vessel, see how blur hides thin walls, and measure the
wall of a synthetic left atrium and its distance to the esophagus.

| Part | What you do | Data |
| --- | --- | --- |
| A | Hounsfield units and iodine (eq-m51-hu, eq-m51-iodine) | NIST attenuation tables |
| B | Projections and the central-slice theorem (eq-m51-radon, eq-m51-slice) | our own phantom |
| C | Filtered back-projection (eq-m51-ramlak, eq-m51-fbp) | our own phantom |
| D | Quantum noise and dose (eq-m51-noise) | our own phantom |
| E | The gating window and motion blur (eq-m51-gating, eq-m51-smear) | published scanner and motion values |
| F | Thin walls under blur (eq-m51-wall) | published wall thickness range |
| G | Wall thickness and esophagus distance on a 3-D left atrium | our own phantom |

Attenuation values are for one photon energy, 60 keV, from NIST Standard Reference Database 126, as in m49; real
CT beams are polyenergetic, so clinical Hounsfield values differ. Gating values come from Petersilka et al. 2008
and Mergen et al. 2023, coronary motion from Achenbach et al. 2000, and left atrial wall thickness from Beinart et
al. 2011. Phantom shapes, the iodine concentration, photon counts and the scanner blur are illustrative. Each part
ends with **checks**; the lab runs in about half a minute and needs no downloads.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import ct

## Part A · Hounsfield units and iodine

CT measures the linear attenuation coefficient $\mu$ in each voxel and reports it on a scale fixed by water and
air: $\mathrm{HU} = 1000\,(\mu - \mu_w)/(\mu_w - \mu_a)$ (eq-m51-hu). Air attenuates about a thousand times less
than water, so we take $\mu_a = 0$. Attenuation adds by mass, so iodine dissolved in blood at $c$ mg/mL adds
$1000\,c \cdot 10^{-3}\,(\mu/\rho)_I / \mu_w$ (eq-m51-iodine).

In [ ]:
for name in ["air", "water", "adipose", "soft tissue", "blood", "cortical bone"]:
    print(f"{name:14s} mu = {ct.mu_per_mm(name):.5f} /mm   {float(ct.hounsfield(ct.mu_per_mm(name))):8.1f} HU")
per = ct.iodine_hu_per_mg_ml()
print(f"iodine adds {per:.1f} HU per mg/mL at 60 keV")
c = np.linspace(0, 15, 31)
plt.plot(c, ct.hounsfield(ct.blood_with_iodine_mu(c)))
plt.axhline(float(ct.hounsfield(ct.mu_per_mm("soft tissue"))), color="gray", ls="--", label="soft tissue")
plt.xlabel("iodine in blood (mg/mL, illustrative)"); plt.ylabel("blood (HU)"); plt.legend(); plt.show()

In [ ]:
# Checks
assert abs(float(ct.hounsfield(ct.mu_per_mm("water")))) < 1e-9
assert float(ct.hounsfield(ct.mu_per_mm("air"))) == -1000
assert abs(per - 36.81) < 0.01
# without contrast, blood and soft tissue differ by only a few HU at this energy
assert abs(float(ct.hounsfield(ct.mu_per_mm("blood")) - ct.hounsfield(ct.mu_per_mm("soft tissue")))) < 10

Without contrast, blood and myocardium differ by about 5 HU here, which is why the chambers are
filled with iodinated contrast before a CT for mapping.

## Part B · Projections and the central-slice theorem

One detector reading is a line integral, $p_\theta(s) = \iint f(x, y)\,\delta(x\cos\theta + y\sin\theta - s)\,dx\,dy$
(eq-m51-radon). For an ellipse this has a closed form, its chord length times its value, so the whole sinogram of
our phantom is exact. The central-slice theorem says the 1-D Fourier transform of one projection is the 2-D
Fourier transform of the image along the line through the origin at the same angle (eq-m51-slice).

In [ ]:
ph = ct.thorax_phantom()
img, x = ct.raster(ph, n=256, oversample=3)
s = ct.detector_axis(320.0, 256)
th = ct.angles(180)
sino = ct.radon_ellipses(ph, th, s)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(ct.hounsfield(img), origin="lower", cmap="gray", vmin=-200, vmax=600, extent=[x[0], x[-1], x[0], x[-1]])
ax[0].set_title("phantom (HU)"); ax[0].set_xlabel("x (mm), patient's left"); ax[0].set_ylabel("y (mm), front")
ax[1].imshow(sino, aspect="auto", cmap="gray", extent=[s[0], s[-1], 180, 0])
ax[1].set_title("sinogram"); ax[1].set_xlabel("s (mm)"); ax[1].set_ylabel("angle (degrees)")
plt.tight_layout(); plt.show()

# every projection holds the same total: the integral of the image
totals = sino.sum(axis=1) * (s[1] - s[0])
print("spread of the projection totals:", f"{totals.std() / totals.mean():.2e}")

# central slice: 1-D transform of one projection against the 2-D transform of the image
X, Y = np.meshgrid(x, x)
dA = (x[1] - x[0]) ** 2
theta = th[40]
ks = np.linspace(0.002, 0.03, 8)
P = np.array([np.sum(sino[40] * np.exp(-2j * np.pi * k * s)) * (s[1] - s[0]) for k in ks])
F = np.array([np.sum(img * np.exp(-2j * np.pi * k * (X * np.cos(theta) + Y * np.sin(theta)))) * dA for k in ks])
rel = np.abs(P - F).max() / np.abs(F).max()
print(f"largest difference between the two transforms: {rel:.3%} of the largest value")

In [ ]:
# Checks
assert totals.std() / totals.mean() < 2e-3
assert rel < 0.01

## Part C · Filtered back-projection

Smearing each projection back along its lines and adding them up (back-projection) blurs the image: low spatial
frequencies are sampled more densely than high ones. The central-slice theorem says how to undo it: weight each
projection's spectrum by $|k|$, the ramp, then back-project (eq-m51-fbp). Sampled every $d$ mm and limited to the
Nyquist frequency, the ramp becomes the Ram-Lak kernel $h[0] = 1/(4d^2)$, $h[n] = -1/(\pi^2 n^2 d^2)$ for odd $n$
and 0 for even $n$ (eq-m51-ramlak).

In [ ]:
rec = ct.fbp(sino, th, s, x)
plain = ct.fbp(sino, th, s, x, filtered=False)
body = img > 0.5 * ct.mu_per_mm("soft tissue")

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(plain, origin="lower", cmap="gray", extent=[x[0], x[-1], x[0], x[-1]])
ax[0].set_title("plain back-projection (arbitrary units)")
ax[1].imshow(ct.hounsfield(rec), origin="lower", cmap="gray", vmin=-200, vmax=600, extent=[x[0], x[-1], x[0], x[-1]])
ax[1].set_title("filtered back-projection (HU)")
plt.tight_layout(); plt.show()

counts, med_err = [12, 30, 60, 120, 180, 360], []
for K in counts:
    tk = ct.angles(K)
    r = ct.fbp(ct.radon_ellipses(ph, tk, s), tk, s, x)
    med_err.append(np.median(np.abs(ct.hounsfield(r) - ct.hounsfield(img))[body]))
plt.loglog(counts, med_err, "o-")
plt.xlabel("projection angles over 180 degrees"); plt.ylabel("median error in the body (HU)"); plt.show()

def hu_at(image, xmm, ymm):
    return float(ct.hounsfield(image[np.argmin(abs(x - ymm)), np.argmin(abs(x - xmm))]))

print(f"left atrium center: {hu_at(rec, 0, -32):.0f} HU (true {hu_at(img, 0, -32):.0f})")
print(f"vertebral body:     {hu_at(rec, 0, -92):.0f} HU (true {hu_at(img, 0, -92):.0f})")

In [ ]:
# Checks
assert abs(hu_at(rec, 0, -32) - hu_at(img, 0, -32)) < 20
assert med_err[0] > 3 * med_err[-2]          # few angles leave streaks
assert all(a >= b for a, b in zip(med_err[:-1], med_err[1:]))

## Part D · Quantum noise and dose

Each ray ends in a photon count $N$ with a Poisson distribution of mean $N_0 e^{-p}$. The line integral we
compute, $-\ln(N/N_0)$, then has variance about $1/\bar N = e^{p}/N_0$ (eq-m51-noise): rays through more tissue
are noisier, and the noise in the image falls as $1/\sqrt{N_0}$. Dose is proportional to $N_0$ (m49), so halving
the noise costs four times the dose. Photon counts here are illustrative.

In [ ]:
rng = np.random.default_rng(51)
th90 = ct.angles(90)
clean = ct.radon_ellipses(ph, th90, s)
base = ct.hounsfield(ct.fbp(clean, th90, s, x))          # the same reconstruction without noise
la = (np.hypot(*np.meshgrid(x - 0.0, x + 32.0)) < 8)     # a disk inside the left atrium
photons = [1e4, 4e4, 1.6e5, 6.4e5]
noise_hu = []
for n0 in photons:
    r = ct.fbp(ct.noisy_sinogram(clean, n0, rng), th90, s, x)
    noise_hu.append(float(np.std((ct.hounsfield(r) - base)[la])))
    print(f"{n0:9.0f} photons per ray: noise in the left atrium {noise_hu[-1]:6.1f} HU")
slope = np.polyfit(np.log(photons), np.log(noise_hu), 1)[0]
print(f"slope of log noise against log photons: {slope:.2f} (theory -0.5)")

In [ ]:
# Checks
assert abs(slope + 0.5) < 0.08

## Part E · The gating window and motion blur

The heart moves, so cardiac CT uses only data from a short window in the cardiac cycle, chosen from the ECG. A
half-scan needs data from about half a rotation; a scanner with two tube-detector pairs 90 degrees apart needs
about a quarter. So the window is about $T_{\text{rot}}/(2 N_{\text{src}})$ (eq-m51-gating), an approximation we
check against published values: 125 ms with one source and 66 ms with two at a 0.25-s rotation (Mergen et al.
2023), and 83 ms with two at 0.33 s (Petersilka et al. 2008). A structure moving at speed $v$ smears over
$v\,\Delta t$; a disk smeared this way has the closed form of eq-m51-smear. Coronary arteries moved in plane at
46.6 mm/s on average, fastest for the right coronary artery (69.5 mm/s) and slowest, 27.9 mm/s, at 48% of the
cycle (Achenbach et al. 2000). The vessel diameter, 3 mm, is illustrative.

In [ ]:
for rot, src, reported in ct.GATING_REPORTED:
    w = ct.temporal_window(rot, src)
    print(f"rotation {rot:.2f} s, {src} source(s): model {1000 * w:.1f} ms, reported {1000 * reported:.0f} ms")

print()
for name, v in ct.CORONARY_SPEED.items():
    b1 = ct.motion_blur_mm(v, ct.temporal_window(0.25, 1))
    b2 = ct.motion_blur_mm(v, ct.temporal_window(0.25, 2))
    print(f"{name:34s} {v:5.1f} mm/s: blur {b1:4.1f} mm (one source), {b2:4.1f} mm (two)")

xs = np.linspace(-8, 8, 801)
for v, style in [(0.0, "--"), (27.9, "-"), (46.6, "-"), (69.5, "-")]:
    b = ct.motion_blur_mm(v, ct.temporal_window(0.25, 1))
    plt.plot(xs, ct.smeared_disk(xs, 0.0, 3.0, b), style, label=f"{v} mm/s" if v else "at rest")
plt.xlabel("position along the motion (mm)"); plt.ylabel("brightness"); plt.legend(); plt.show()
peak_fast = float(ct.smeared_disk(0.0, 0.0, 3.0, ct.motion_blur_mm(69.5, 0.125)))
print(f"a 3-mm vessel at 69.5 mm/s keeps {peak_fast:.0%} of its brightness with a 125-ms window")

In [ ]:
# Checks
for rot, src, reported in ct.GATING_REPORTED:
    assert abs(ct.temporal_window(rot, src) - reported) / reported < 0.06
assert abs(peak_fast - 3.0 / (69.5 * 0.125)) < 1e-9

Blur scales with the window, so the slowest phase and the second source each help. The constant-speed
model ignores how motion changes within the window and any motion correction a scanner applies.

## Part F · Thin walls under blur

Model the scanner's blur as a Gaussian of full width at half maximum (FWHM) $\Gamma$, standard deviation
$\sigma = \Gamma/(2\sqrt{2\ln 2})$. A wall of thickness $w$ and unit contrast then has the profile
$\tfrac12[\operatorname{erf}((x + w/2)/(\sqrt2\sigma)) - \operatorname{erf}((x - w/2)/(\sqrt2\sigma))]$, with
peak $\operatorname{erf}(w/(2\sqrt2\sigma))$ (eq-m51-wall). Read thickness with a fixed threshold at half the
contrast and thin walls vanish; read it as the width at half the profile's own peak and they never look thinner
than $\Gamma$. Left atrial walls on CT measured 1.89 mm on average, from 0.5 to 3.5 mm (Beinart et al. 2011). The
blur values are illustrative.

In [ ]:
widths = np.linspace(0.2, 4.0, 77)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
for fwhm in (0.5, 1.0, 2.0):
    sg = ct.fwhm_to_sigma(fwhm)
    ax[0].plot(widths, [ct.slab_peak(w, sg) for w in widths], label=f"blur FWHM {fwhm} mm")
    ax[1].plot(widths, [ct.threshold_thickness(w, sg) for w in widths], label=f"threshold, FWHM {fwhm}")
    ax[1].plot(widths, [ct.fwhm_thickness(w, sg) for w in widths], ":", label=f"FWHM reading, {fwhm}")
ax[1].plot(widths, widths, "k--", lw=1, label="true")
for a in ax:
    a.axvspan(ct.LA_WALL_MM["min"], ct.LA_WALL_MM["max"], color="0.9", zorder=0)
ax[0].set_xlabel("wall thickness (mm)"); ax[0].set_ylabel("peak contrast (fraction)"); ax[0].legend()
ax[1].set_xlabel("wall thickness (mm)"); ax[1].set_ylabel("measured (mm)"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

sg1 = ct.fwhm_to_sigma(1.0)
for w in (ct.LA_WALL_MM["min"], ct.LA_WALL_MM["mean"], ct.LA_WALL_MM["max"]):
    print(f"wall {w:.2f} mm, blur 1 mm: peak {ct.slab_peak(w, sg1):.0%}, threshold reads "
          f"{ct.threshold_thickness(w, sg1):.2f} mm, FWHM reads {ct.fwhm_thickness(w, sg1):.2f} mm")

In [ ]:
# Checks
assert ct.threshold_thickness(0.5, sg1) == 0.0                 # the thinnest published wall vanishes at 1-mm blur
assert ct.fwhm_thickness(0.5, sg1) > 1.0                       # or reads thicker than the blur
assert abs(ct.threshold_thickness(3.5, sg1) - 3.5) < 0.01      # the thickest one reads right

## Part G · Wall thickness and esophagus distance on a 3-D left atrium

A contrast-filled ellipsoidal left atrium sits in fat. Its wall is 0.5 mm thick at the back and 3.5 mm at the
front, the published range (Beinart et al. 2011); an esophagus runs 3 mm behind the back wall. The esophagus
touched the mid posterior left atrium in every patient in a CT study of its course (Tsao et al. 2005), which is
why its distance matters. We segment the wall with a window between the midpoints of fat and wall and of wall and
blood, measure it along rays from the center, and repeat after blurring and with coarser voxels. Shapes, the
blood value (400 HU) and the blur are illustrative.

In [ ]:
vol, xv, masks = ct.atrium_volume(voxel=0.5, wall=(0.5, 3.5))
fat_hu = float(ct.hounsfield(ct.mu_per_mm("adipose")))
wall_hu = float(ct.hounsfield(ct.mu_per_mm("soft tissue")))
lo, hi = (fat_hu + wall_hu) / 2, (wall_hu + 400.0) / 2
rng = np.random.default_rng(7)
dirs = ct.sphere_directions(300, rng)
truth = ct.true_ray_thickness(dirs, wall=(0.5, 3.5))

def measure(v, axis):
    return ct.ray_thickness((v > lo) & (v < hi), axis, dirs)

rows = [("0.5 mm voxels, no blur", measure(vol, xv))]
for fwhm in (1.0, 2.0):
    rows.append((f"0.5 mm voxels, blur {fwhm} mm", measure(ct.blur_volume(vol, 0.5, fwhm), xv)))
coarse, xc = ct.downsample(vol, xv, 3)
rows.append(("1.5 mm voxels (block average)", measure(coarse, xc)))

thin, thick = truth < 1.0, truth > 3.0
print(f"{'median thickness (mm)':32s} {'walls < 1 mm':>14s} {'walls > 3 mm':>14s} {'rays > 6 mm':>12s}")
print(f"{'true':32s} {np.median(truth[thin]):14.2f} {np.median(truth[thick]):14.2f} {0:12d}")
for name, m in rows:
    print(f"{name:32s} {np.median(m[thin]):14.2f} {np.median(m[thick]):14.2f} {int(np.sum(m > 6)):12d}")
merged = rows[3][1] > 6
print(f"with 1.5 mm voxels, {merged.sum()} rays ran on through the fat gap into the esophagus;"
      f" all point backward: {bool(np.all(dirs[merged, 1] < 0))}")

z0 = np.argmin(abs(xv))
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(vol[:, :, z0].T, origin="lower", cmap="gray", vmin=-200, vmax=600, extent=[xv[0], xv[-1], xv[0], xv[-1]])
ax[0].set_title("mid slice (HU)"); ax[0].set_xlabel("x (mm)"); ax[0].set_ylabel("y (mm), front up")
ax[1].scatter(truth, rows[2][1], s=8, label="blur 2 mm")
ax[1].scatter(truth[~merged], rows[3][1][~merged], s=8, label="1.5 mm voxels (rays > 6 mm not shown)")
ax[1].plot([0, 4], [0, 4], "k--", lw=1)
ax[1].set_ylim(0, 6)
ax[1].set_xlabel("true thickness (mm)"); ax[1].set_ylabel("measured (mm)"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

for fwhm in (1.0, 2.0):
    w0 = ct.step_window_width(ct.fwhm_to_sigma(fwhm), lo, hi, 400.0, fat_hu)
    print(f"blur {fwhm} mm: a blood-to-fat step with no wall at all would read {w0:.2f} mm")

gap = ct.min_distance_mm(masks["outer"], masks["esophagus"], xv)
print(f"esophagus to the outer wall: {gap:.2f} mm (built 3.0 mm)")

In [ ]:
# Checks
assert abs(np.median(rows[0][1][thick]) - np.median(truth[thick])) < 0.3
assert np.median(rows[3][1][thin]) > np.median(truth[thin]) + 0.3      # coarse voxels inflate the thinnest walls
assert merged.sum() > 0 and np.all(dirs[merged, 1] < 0)                  # and merge wall and esophagus
w0_2mm = ct.step_window_width(ct.fwhm_to_sigma(2.0), lo, hi, 400.0, fat_hu)
assert abs(np.median(rows[2][1][thin]) - w0_2mm) < 0.2                   # at 2-mm blur thin walls read as the step
assert abs(gap - 3.0) <= 0.5

Where the wall is thicker than the blur, a threshold window between tissue midpoints measures it well.
Where it is thinner, what you read depends on the blur, the voxel size and the thresholds, not only on the wall.
Here the wall separates blood from fat, so the blurred step from one to the other passes through the wall's window
even where the wall is too thin to show its own contrast; with 2 mm of blur the thinnest walls read about as wide
as that step alone.
With coarse voxels, partial volume can even bridge the fat between the back wall and the esophagus, so the two
read as one structure. This is one reason published wall thickness and distance values depend on the scanner and
the method.

## Where to go next

- In part C, add noise from part D and compare the ramp with a smoother filter (multiply the ramp by a window).
- In part E, how short must the window be to keep 80% of the brightness of a 3-mm right coronary artery?
- In part F, at what blur does the mean published wall (1.89 mm) fall below half its contrast?
- In part G, move the esophagus to 1 mm and blur by 2 mm. Can a threshold still separate it from the wall?